## TEst

In [ ]:
import requests
import pandas as pd
import os

from dotenv import load_dotenv

# =========================================
# 1. .env 로드
# =========================================

load_dotenv()

CONSUMER_KEY = os.getenv("SGIS_CONSUMER_KEY")
CONSUMER_SECRET = os.getenv("SGIS_CONSUMER_SECRET")

# =========================================
# 2. AccessToken 발급
# =========================================

AUTH_URL = "https://sgisapi.kostat.go.kr/OpenAPI3/auth/authentication.json"

auth_params = {
    "consumer_key": CONSUMER_KEY,
    "consumer_secret": CONSUMER_SECRET
}

auth_res = requests.get(AUTH_URL, params=auth_params)

auth_json = auth_res.json()

if auth_json["errCd"] != 0:
    raise Exception(auth_json["errMsg"])

ACCESS_TOKEN = auth_json["result"]["accessToken"]

print("AccessToken 발급 성공")

# =========================================
# 3. 시도 코드
# =========================================

sido_codes = {
    "서울": "11",
    "부산": "21",
    "대구": "22",
    "인천": "23",
    "광주": "24",
    "대전": "25",
    "울산": "26",
    "세종": "29",
    "경기": "31",
    "강원": "32",
    "충북": "33",
    "충남": "34",
    "전북": "35",
    "전남": "36",
    "경북": "37",
    "경남": "38",
    "제주": "39"
}

# =========================================
# 4. 산업분류 코드 조회
# =========================================

industry_url = "https://sgisapi.kostat.go.kr/OpenAPI3/stats/industrycode.json"

industry_params = {
    "accessToken": ACCESS_TOKEN,
    "class_deg": "10"
}

industry_res = requests.get(
    industry_url,
    params=industry_params
)

industry_json = industry_res.json()

industry_df = pd.DataFrame(
    industry_json["result"]
)

print(industry_df.head())

# =========================================
# 5. 사업체 통계 API
# =========================================

company_url = "https://sgisapi.kostat.go.kr/OpenAPI3/stats/company.json"

all_result = []

industry_sample = industry_df.head(20)

for sido_name, sido_cd in sido_codes.items():

    print(f"\n[{sido_name}] 수집 중...")

    region_result = []

    for _, row in industry_sample.iterrows():

        class_code = row["class_code"]
        class_nm = row["class_nm"]

        params = {
            "accessToken": ACCESS_TOKEN,
            "year": "2019",
            "adm_cd": sido_cd,
            "low_search": "0",
            "class_code": class_code
        }

        res = requests.get(
            company_url,
            params=params
        )

        try:
            data = res.json()

        except:
            continue

        # API 오류
        if data["errCd"] != 0:
            continue

        # 데이터 없음
        if len(data["result"]) == 0:
            continue

        # 사업체 수 추출
        corp_cnt = pd.to_numeric(
            data["result"][0]["corp_cnt"],
            errors="coerce"
        )

        # N/A 제거
        if pd.isna(corp_cnt):
            continue

        corp_cnt = int(corp_cnt)

        region_result.append({
            "시도": sido_name,
            "산업명": class_nm,
            "사업체수": corp_cnt
        })

    region_df = pd.DataFrame(region_result)

    if region_df.empty:
        continue

    # 사업체 수 기준 TOP5
    top5 = (
        region_df
        .sort_values(
            "사업체수",
            ascending=False
        )
        .head(5)
    )

    all_result.append(top5)

# =========================================
# 6. 결과 통합
# =========================================

final_df = pd.concat(all_result)

print("\n===== 시도별 사업체수 TOP5 산업 =====")
print(final_df)

# =========================================
# 7. CSV 저장
# =========================================

final_df.to_csv(
    "시도별_산업_TOP5.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV 저장 완료")

AccessToken 발급 성공
                  class_nm class_code
0              농업, 임업 및 어업          A
1                       광업          B
2                      제조업          C
3   전기, 가스, 증기 및 공기 조절 공급업          D
4  수도, 하수 및 폐기물 처리, 원료 재생업          E

[서울] 수집 중...

[부산] 수집 중...

[대구] 수집 중...

[인천] 수집 중...

[광주] 수집 중...

[대전] 수집 중...


ValueError: invalid literal for int() with base 10: 'N/A'